In [1]:
# get openai model
# load faq dataset
# get 50 samples
# generate eval dataset for faq
# persist to data/eval

In [2]:
import sys
sys.path.append("../../")
import json
import random

from models.generation import openai_generate
from ingestion.faq.faq_chunker import split_by_docs

In [4]:
# load faq dataset
raw_docs = split_by_docs("../../data/raw/faq.docx")

docs_by_section_id = {}
for doc in raw_docs:
    grouped = docs_by_section_id.setdefault(doc["section_id"], {
        "section_id": doc["section_id"],
        "header": doc["header"],
        "content": [],
    })
    grouped["content"].append(doc["content"])

docs = list(docs_by_section_id.values())
docs[:100]

[{'section_id': 1,
  'header': 'How to register on HopShop',
  'content': ['If you want to be an active user of HopShop, register in a few simple steps. It is free of charge.\nCheck how to register\nwith an email address\nwith a Google or Facebook account\nwith a phone number.']},
 {'section_id': 2,
  'header': 'How to register with an email',
  'content': ['Open the\xa0registration page. Choose whether you want to hold a\xa0regular account or a business account.\nEnter your email address and create a password.\nDetermine whether you are over 18 years old. If not, provide your exact date of birth. You will hold a\xa0Junior account\xa0which will be transformed into a regular account once you turn 18.\nAccept the\xa0HopShop Terms & Conditions\xa0and click [register].\nYou will receive a confirmation email. Open your mailbox and click (confirm registration) to complete the registration process.\nOnce you complete the registration, we will redirect you to your new HopShop account. We will 

In [4]:
def make_prompt(header, content, section_id):
    return f"""
You are generating evaluation examples for a FAQ retrieval system.

Task:
Create exactly ONE user question based on the provided FAQ.

Input:
- Header: {header}
- Content: {content}
- Section_Id: {section_id}

Requirements:
- The question must reflect a realistic user query.
- It must be fully grounded in the Header and Content.
- Do NOT introduce any new facts.
- You may paraphrase the intent, but preserve meaning.
- Prefer queries that a real user would type in a search bar.
- The question should be concise and specific.
- Avoid overly generic or vague wording.
- Return ONLY ONE question.

Output format:
- Return ONLY valid JSON.
- Do NOT include any explanations or extra text.
- Ensure the JSON is syntactically correct (escape quotes if needed).

Schema:
{{
  "question": "<generated question>",
  "reference_answer": "<Content>",
  "relevant_docs": [{section_id}],
  "contexts": ["<Content>"]
}}
"""

In [5]:
def make_adversarial_answer_prompt(question, header, content, section_id):
    return f"""
You are generating answer-level evaluation data for groundedness and answer relevance.

Task:
Generate exactly ONE plausible-looking answer to the user question that is NOT fully grounded in the FAQ content.

Input:
- Question: {question}
- Header: {header}
- Content: {content}
- Section_Id: {section_id}

Requirements:
- The answer must look realistic and mostly relevant to the question.
- The answer must contain 1 or 2 subtle grounding errors.
- Allowed error types:
  - slightly wrong number, date, limit, or condition
  - adding a plausible but unsupported detail
  - overstating what the FAQ says
  - omitting an important restriction and making the answer too broad
  - mild contradiction of the FAQ content
- The answer should NOT be obviously absurd or unrelated.
- It should be difficult to detect without checking the source content.
- Return ONLY valid JSON.

Schema:
{{
  "question": "{question}",
  "reference_answer": "{content}",
  "relevant_docs": [{section_id}],
  "contexts": ["{content}"],
  "candidate_answer": "<subtly ungrounded answer>",
  "answer_type": "adversarial",
  "grounded_label": 0,
  "relevance_label": 1
}}
"""

In [6]:
def make_bad_answer_prompt(question, header, content, section_id):
    return f"""
You are generating answer-level evaluation data for groundedness and answer relevance.

Task:
Generate exactly ONE incorrect answer to the user question that is not grounded in the FAQ content.

Input:
- Question: {question}
- Header: {header}
- Content: {content}
- Section_Id: {section_id}

Requirements:
- The answer must be clearly unsupported by the FAQ content.
- It may contradict the FAQ content or invent details not present there.
- It should still be phrased as a natural answer to the question.
- Do not make it nonsensical; it should remain readable and on-topic.
- Return ONLY valid JSON.

Schema:
{{
  "question": "{question}",
  "reference_answer": "{content}",
  "relevant_docs": [{section_id}],
  "contexts": ["{content}"],
  "candidate_answer": "<incorrect ungrounded answer>",
  "answer_type": "bad",
  "grounded_label": 0,
  "relevance_label": 0
}}
"""

In [7]:
rows = []
calibr_rows = []

sampled_docs = random.sample(docs, 50)

for ob in sampled_docs:
    content = "\n\n".join(ob["content"])
    base_prompt = make_prompt(ob['header'], content, ob["section_id"])
    resp = openai_generate(base_prompt)
    test = parse_model_json(resp)

    adv_prompt = make_adversarial_answer_prompt(test["question"], ob['header'], content, ob["section_id"])
    adv_resp = openai_generate(adv_prompt)
    calibr_rows.append(parse_model_json(adv_resp))

    bad_prompt = make_bad_answer_prompt(test["question"], ob['header'], content, ob["section_id"])
    bad_resp = openai_generate(bad_prompt)
    calibr_rows.append(parse_model_json(bad_resp))
    calibr_rows.append({
        "question": test["question"],
        "reference_answer": content,
        "relevant_docs": [ob["section_id"]],
        "contexts": ob["content"],
        "candidate_answer": test["reference_answer"],
        "answer_type": "good",
        "grounded_label": 1,
        "relevance_label": 1
    })


    rows.append(test)

In [8]:
with open("../data/eval/faq_evaluation_dataset.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        row = parse_model_json(row)

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [10]:
calibr_rows

['{\n  "question": "Will I get a full refund including shipping if I return my order?",\n  "reference_answer": "The seller refunds you for the returned products and the cheapest delivery option. However, you may not receive the full amount you paid for the purchase.\\nIn some cases, the seller may reduce the refund amount ― for example, if the product has been damaged in transport or has clear signs of use. If the seller refunds you only partially, you will find a justification for that in an email.\\nThe seller is not obliged to refund you for all the additional expenses. For example, if you choose a different delivery option than the cheapest one offered by the seller, they will refund you the cost of the cheapest option.",\n  "relevant_docs": [83],\n  "contexts": ["The seller refunds you for the returned products and the cheapest delivery option. However, you may not receive the full amount you paid for the purchase.\\nIn some cases, the seller may reduce the refund amount ― for exa

In [12]:
with open("../data/eval/faq_answer_calibration_dataset.jsonl", "w", encoding="utf-8") as f:
    for row in calibr_rows:
        row = parse_model_json(row)

        f.write(json.dumps(row, ensure_ascii=False) + "\n")